# 📘 Cleaned piNEUMA Parser for Local Folder
This notebook loads and parses **all piNEUMA CSV files** from your local `data/` folder.

In [ ]:
# ✅ Step 1: Import libraries
import pandas as pd
import numpy as np
import os
import glob

In [ ]:
# ✅ Step 2: Set path to your local folder
folder_path = '../data'  # path relative to notebooks folder
csv_files = glob.glob(os.path.join(folder_path, '*.csv'))
dataframes = {}

In [ ]:

# ✅ Step 3: Parse each file into a labeled DataFrame
for file_path in csv_files:
    filename = os.path.basename(file_path)
    parts = filename.split('_')

    if len(parts) < 3:
        print(f"Skipping file with unexpected format: {filename}")
        continue

    date_str = parts[0]
    location_str = parts[1]
    start_time_str = parts[2]
    monthday = date_str[4:]

    loc_num = location_str.replace("d", "")
    loc_label = f"L{loc_num}"

    if start_time_str.startswith("0"):
        start_time_str = start_time_str[1:]

    var_name = f"df_{loc_label}{start_time_str}_{monthday}"

    rows = []
    with open(file_path, 'r', encoding='utf-8') as f:
        next(f)
        for line_num, line in enumerate(f, start=2):
            if line_num > 5000:
                break
            fields = line.strip().split(';')
            if len(fields) < 4:
                continue

            track_id    = fields[0]
            vehicle_type = fields[1]
            traveled_d  = fields[2]
            avg_speed   = fields[3]
            time_step_fields = fields[4:]

            while time_step_fields and not time_step_fields[-1].strip():
                time_step_fields.pop()

            leftover = len(time_step_fields) % 6
            if leftover != 0:
                time_step_fields = time_step_fields[:-leftover]

            n_blocks = len(time_step_fields) // 6

            for i in range(n_blocks):
                offset = i * 6
                lat     = time_step_fields[offset + 0]
                lon     = time_step_fields[offset + 1]
                speed   = time_step_fields[offset + 2]
                lon_acc = time_step_fields[offset + 3]
                lat_acc = time_step_fields[offset + 4]
                tstamp  = time_step_fields[offset + 5]
                rows.append({
                    "track_id":   track_id,
                    "type":       vehicle_type,
                    "traveled_d": traveled_d,
                    "avg_speed":  avg_speed,
                    "lat":        lat,
                    "lon":        lon,
                    "speed":      speed,
                    "lon_acc":    lon_acc,
                    "lat_acc":    lat_acc,
                    "time":       tstamp,
                    "time_step":  i
                })

    df = pd.DataFrame(rows)
    numeric_cols = ["traveled_d", "avg_speed", "lat", "lon", "speed", "lon_acc", "lat_acc", "time"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    dataframes[var_name] = df
    print(f"Parsed {filename} -> {var_name} with shape: {df.shape}")


In [ ]:
# ✅ Step 4: Check one example
list(dataframes.keys())

In [ ]:
# ✅ Step 5: Preview one of the parsed DataFrames
sample_key = list(dataframes.keys())[0]
dataframes[sample_key].head()